# RAG Pipeline for Industrial Inspection Rules

This notebook builds the retrieval layer for VisionGuard AI.

The goal is to retrieve relevant inspection rules based on a user query or VLM-generated image description.

Pipeline:
1. Load inspection rule documents
2. Split rules into structured chunks
3. Generate embeddings using Sentence Transformers
4. Store embeddings in FAISS
5. Retrieve top-k relevant rules for a query

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

d:\visionguard-ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##  Project paths

We define the paths for rule documents and vector store output.

In [2]:
PROJECT_ROOT = Path("..").resolve()

RULES_DIR = PROJECT_ROOT / "data" / "inspection_rules"
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store"

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

FAISS_INDEX_PATH = VECTOR_STORE_DIR / "faiss_index.bin"
CHUNKS_PATH = VECTOR_STORE_DIR / "rule_chunks.json"

print("Project root:", PROJECT_ROOT)
print("Rules directory:", RULES_DIR)
print("Vector store directory:", VECTOR_STORE_DIR)

Project root: D:\visionguard-ai
Rules directory: D:\visionguard-ai\data\inspection_rules
Vector store directory: D:\visionguard-ai\vector_store


## Rule files

Each markdown file contains industrial inspection rules.
We load all `.md` files from the inspection rules directory.

In [3]:
def load_rule_documents(rules_dir: Path) -> list[dict]:
    documents = []

    if not rules_dir.exists():
        raise FileNotFoundError(f"Rules directory not found: {rules_dir}")

    for file_path in rules_dir.glob("*.md"):
        content = file_path.read_text(encoding="utf-8")

        documents.append({
            "filename": file_path.name,
            "content": content
        })

    if not documents:
        raise ValueError("No markdown rule documents found.")

    return documents


documents = load_rule_documents(RULES_DIR)

print(f"Loaded {len(documents)} rule documents:")
for doc in documents:
    print("-", doc["filename"], "|", len(doc["content"]), "characters")

Loaded 4 rule documents:
- defect_rules.md | 2555 characters
- escalation_policy.md | 1745 characters
- machine_safety_rules.md | 2691 characters
- ppe_rules.md | 2872 characters


## Split documents into rule chunks

Instead of chunking blindly every few hundred words, we split by `## Rule ID`.

This keeps each inspection rule as one meaningful retrieval unit.

In [4]:
def extract_rule_chunks(documents: list[dict]) -> list[dict]:
    chunks = []

    for doc in documents:
        filename = doc["filename"]
        content = doc["content"]

        # Split document by each rule heading
        raw_rules = re.split(r"(?=## Rule ID:)", content)

        for raw_rule in raw_rules:
            raw_rule = raw_rule.strip()

            if not raw_rule.startswith("## Rule ID:"):
                continue

            rule_id_match = re.search(r"Rule ID:\s*([A-Z]+-\d+)", raw_rule)
            category_match = re.search(r"Category:\s*(.*)", raw_rule)
            severity_match = re.search(r"Severity:\s*(.*)", raw_rule)

            rule_id = rule_id_match.group(1).strip() if rule_id_match else "UNKNOWN"
            category = category_match.group(1).strip() if category_match else "Unknown"
            severity = severity_match.group(1).strip() if severity_match else "Unknown"

            chunks.append({
                "rule_id": rule_id,
                "category": category,
                "severity": severity,
                "source_file": filename,
                "text": raw_rule
            })

    if not chunks:
        raise ValueError("No rule chunks extracted. Check markdown formatting.")

    return chunks


rule_chunks = extract_rule_chunks(documents)

print(f"Extracted {len(rule_chunks)} rule chunks.")

for chunk in rule_chunks[:3]:
    print("\n---")
    print("Rule ID:", chunk["rule_id"])
    print("Category:", chunk["category"])
    print("Severity:", chunk["severity"])
    print("Source:", chunk["source_file"])
    print(chunk["text"][:300])

Extracted 30 rule chunks.

---
Rule ID: DEF-001
Category: Surface Crack
Severity: High
Source: defect_rules.md
## Rule ID: DEF-001
Category: Surface Crack
Inspection Area: Metal, Plastic, or Composite Component
Requirement: Components must not show visible cracks, fractures, or structural breaks.
Severity: High
Recommended Action: Reject the component and send it for detailed quality inspection.
Human Review

---
Rule ID: DEF-002
Category: Surface Scratch
Severity: Medium
Source: defect_rules.md
## Rule ID: DEF-002
Category: Surface Scratch
Inspection Area: Painted or Finished Surface
Requirement: Finished components must not show deep scratches, coating damage, or visible surface marks beyond tolerance.
Severity: Medium
Recommended Action: Mark the component for rework or quality review de

---
Rule ID: DEF-003
Category: Missing Component
Severity: High
Source: defect_rules.md
## Rule ID: DEF-003
Category: Missing Component
Inspection Area: Assembly Line
Requirement: Assembled product

## Load embedding model

We use `all-MiniLM-L6-v2`, a lightweight sentence embedding model.

It is fast and good enough for semantic retrieval in this project.

In [5]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

d:\visionguard-ai\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harit\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7833.28it/s]


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


## Generate embeddings

Each rule chunk is converted into a numerical vector.
FAISS will use these vectors for similarity search.

In [6]:
rule_texts = [chunk["text"] for chunk in rule_chunks]

embeddings = embedding_model.encode(
    rule_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]

Embedding shape: (30, 384)


## Build FAISS index

FAISS stores the rule embeddings and allows fast similarity search.
We use cosine similarity by normalizing vectors and applying inner product search.

In [7]:
# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index created.")
print("Number of vectors in index:", index.ntotal)
print("Embedding dimension:", embedding_dim)

FAISS index created.
Number of vectors in index: 30
Embedding dimension: 384


## Test retrieval

We now test whether the system retrieves correct rules for realistic inspection queries.

In [8]:
def retrieve_rules(query: str, top_k: int = 3) -> list[dict]:
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = rule_chunks[idx].copy()
        chunk["score"] = float(score)
        results.append(chunk)

    return results

## Test with PPE query

In [9]:
query = "worker is not wearing helmet near operating machinery"

results = retrieve_rules(query, top_k=3)

print("Query:", query)

for result in results:
    print("\n---")
    print("Score:", round(result["score"], 4))
    print("Rule ID:", result["rule_id"])
    print("Category:", result["category"])
    print("Severity:", result["severity"])
    print("Source:", result["source_file"])
    print(result["text"])

Query: worker is not wearing helmet near operating machinery

---
Score: 0.6079
Rule ID: PPE-001
Category: Head Protection
Severity: High
Source: ppe_rules.md
## Rule ID: PPE-001
Category: Head Protection
Inspection Area: Active Machine Zone
Requirement: Workers must wear safety helmets when working near operating machinery, moving equipment, or overhead hazard areas.
Severity: High
Recommended Action: Stop work temporarily and ensure the worker wears an approved safety helmet before resuming operations.
Human Review Required: Yes

---
Score: 0.406
Rule ID: MACH-005
Category: Unsafe Worker Distance
Severity: High
Source: machine_safety_rules.md
## Rule ID: MACH-005
Category: Unsafe Worker Distance
Inspection Area: Active Machine Zone
Requirement: Workers must maintain safe distance from moving machinery unless authorized and protected by safety controls.
Severity: High
Recommended Action: Move worker away from unsafe zone and verify safe working distance.
Human Review Required: Yes

--

## Test with defect query

In [10]:
query = "metal component has visible crack and structural damage"

results = retrieve_rules(query, top_k=3)

print("Query:", query)

for result in results:
    print("\n---")
    print("Score:", round(result["score"], 4))
    print("Rule ID:", result["rule_id"])
    print("Category:", result["category"])
    print("Severity:", result["severity"])
    print("Source:", result["source_file"])
    print(result["text"])

Query: metal component has visible crack and structural damage

---
Score: 0.5233
Rule ID: DEF-001
Category: Surface Crack
Severity: High
Source: defect_rules.md
## Rule ID: DEF-001
Category: Surface Crack
Inspection Area: Metal, Plastic, or Composite Component
Requirement: Components must not show visible cracks, fractures, or structural breaks.
Severity: High
Recommended Action: Reject the component and send it for detailed quality inspection.
Human Review Required: Yes

---
Score: 0.3576
Rule ID: DEF-002
Category: Surface Scratch
Severity: Medium
Source: defect_rules.md
## Rule ID: DEF-002
Category: Surface Scratch
Inspection Area: Painted or Finished Surface
Requirement: Finished components must not show deep scratches, coating damage, or visible surface marks beyond tolerance.
Severity: Medium
Recommended Action: Mark the component for rework or quality review depending on defect size.
Human Review Required: Yes

---
Score: 0.3412
Rule ID: DEF-006
Category: Deformation
Severity: H

## Test with unclear image query

In [11]:
query = "image is blurry and the inspection evidence is unclear"

results = retrieve_rules(query, top_k=3)

print("Query:", query)

for result in results:
    print("\n---")
    print("Score:", round(result["score"], 4))
    print("Rule ID:", result["rule_id"])
    print("Category:", result["category"])
    print("Severity:", result["severity"])
    print("Source:", result["source_file"])
    print(result["text"])

Query: image is blurry and the inspection evidence is unclear

---
Score: 0.6024
Rule ID: ESC-006
Category: Missing or Incomplete Input
Severity: Review Needed
Source: escalation_policy.md
## Rule ID: ESC-006
Category: Missing or Incomplete Input
Condition: Uploaded image is blurry, too dark, cropped, or does not show enough context for inspection.
Severity: Review Needed
Required Action: Request a clearer image or additional inspection evidence.
Human Review Required: Yes

---
Score: 0.4052
Rule ID: ESC-005
Category: Unclear Visual Evidence
Severity: Review Needed
Source: escalation_policy.md
## Rule ID: ESC-005
Category: Unclear Visual Evidence
Condition: The AI system cannot confidently determine whether a violation or defect is present.
Severity: Review Needed
Required Action: Mark the case for human review and do not make an automatic decision.
Human Review Required: Yes

---
Score: 0.3295
Rule ID: DEF-004
Category: Misalignment
Severity: Medium
Source: defect_rules.md
## Rule ID:

## Save FAISS index and rule chunks

We save the vector index and metadata so the backend can use them later without rebuilding every time.

In [12]:
faiss.write_index(index, str(FAISS_INDEX_PATH))

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(rule_chunks, f, indent=4, ensure_ascii=False)

print("Saved FAISS index to:", FAISS_INDEX_PATH)
print("Saved rule chunks to:", CHUNKS_PATH)

Saved FAISS index to: D:\visionguard-ai\vector_store\faiss_index.bin
Saved rule chunks to: D:\visionguard-ai\vector_store\rule_chunks.json
